# KeelNet Stage 4: Unsupported-Confidence Control Template

Use this notebook as the teammate-ready working file for Stage 4.

Official workflow:

1. open this notebook in browser Google Colab from `stage/04-unsupported-confidence-control`
2. run the setup, config, and validation cells
3. edit code locally in VS Code when needed
4. commit and push changes to GitHub
5. rerun the setup cell so `/content/KeelNet` updates
6. repeat until the stage succeeds


## Stage Notes

### Goal

Reduce confident unsupported answers without collapsing usefulness.

### Scope

- input: answer, support signal, confidence signal
- output: better control over unsupported confident answering

### Main Change

- add an explicit penalty or control mechanism for confident unsupported outputs

### Main Metrics

- unsupported-answer rate
- supported-answer rate
- answer `F1`
- abstain `F1`

### Handoff Condition

Do not move to the next stage until unsupported confident answers go down without answer quality collapsing.


## 1. Setup And Sync

Run this cell in browser Google Colab. It mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out `stage/04-unsupported-confidence-control`, and installs the project.

Path reminder:

- code runs from `/content/KeelNet`
- team artifacts should save to `/content/drive/MyDrive/KeelNet`
- share that `KeelNet` folder with your teammates in Google Drive
- if `DRIVE_PROJECT_DIR` does not start with `/content/drive/`, the path is wrong

Important:

- edit code locally in VS Code if you want
- execute this notebook in browser Colab, not a normal local Jupyter kernel
- push code to GitHub
- rerun this cell before rerunning stage commands so the Colab runtime updates `/content/KeelNet`
                


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


# Team default: save artifacts in a shared folder under MyDrive.
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")

GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
GIT_BRANCH = "stage/04-unsupported-confidence-control"
LOCAL_REPO_DIR = Path("/content/KeelNet")


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def mount_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    if not str(DRIVE_PROJECT_DIR).startswith("/content/drive/"):
        raise ValueError(
            f"DRIVE_PROJECT_DIR must point inside /content/drive, got: {DRIVE_PROJECT_DIR}"
        )
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir: {DRIVE_PROJECT_DIR}")


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def ensure_repo() -> Path:
    if (LOCAL_REPO_DIR / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=LOCAL_REPO_DIR)
    else:
        run_setup(["git", "clone", GIT_REPO_URL, str(LOCAL_REPO_DIR)])

    run_setup(["git", "checkout", GIT_BRANCH], cwd=LOCAL_REPO_DIR)
    run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=LOCAL_REPO_DIR)
    return LOCAL_REPO_DIR


mount_drive()
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime repo dir: {REPO_DIR}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


## 2. Configure The Run

Set `AUTHOR_NAME` to your name. This notebook builds the stage-specific `RUN_NAME` automatically as `yourname-stage4-v1`, `yourname-stage4-v2`, `yourname-stage4-v3`, and so on based on completed runs.

This template creates a small `RUN_STARTED.txt` file in the Drive run folder right away so you can confirm the output path is correct.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys

import torch

REPO_DIR = Path(REPO_DIR).resolve()

# Change only this for each teammate. The notebook builds the stage name and next version automatically.
AUTHOR_NAME = "yourname"
RUN_BASENAME = f"{AUTHOR_NAME}-stage4"
ARTIFACTS_ROOT = DRIVE_PROJECT_DIR / "artifacts" / "stage4_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME

STAGE_TITLE = 'Stage 4: Unsupported-Confidence Control'
STAGE_OBJECTIVE = 'Reduce confident unsupported answers without collapsing usefulness.'
TARGET_METRICS = ['unsupported-answer rate', 'supported-answer rate', 'answer F1', 'abstain F1']
IMPLEMENTATION_HINTS = ['input: answer, support signal, confidence signal', 'output: better control over unsupported confident answering', 'add an explicit penalty or control mechanism for confident unsupported outputs']
SUGGESTED_MODULES = ['keelnet.control', 'keelnet.evaluate', 'keelnet.metrics']

# Fill these in when the stage code is ready.
RUN_SMOKE_TEST = False
SMOKE_TEST_COMMANDS = [
    # Example:
    # [sys.executable, "-m", "keelnet.some_module", "--help"],
]
STAGE_COMMANDS = [
    # Example:
    # [sys.executable, "-m", "keelnet.some_module", "--output-dir", str(OUTPUT_ROOT / "trial-1")],
]

RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"stage={STAGE_TITLE}",
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"repo_dir={REPO_DIR}",
            f"git_branch={CURRENT_BRANCH}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(f"Repo dir: {REPO_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Target metrics:", ", ".join(TARGET_METRICS))
print("Suggested modules:", ", ".join(SUGGESTED_MODULES))


def run(cmd):
    print("$", " ".join(str(part) for part in cmd))
    completed = subprocess.run([str(part) for part in cmd], cwd=REPO_DIR, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()


def run_many(commands, *, label: str) -> bool:
    if not commands:
        print(f"No commands configured for {label} yet.")
        return False

    for index, cmd in enumerate(commands, start=1):
        print(f"\n[{label} {index}/{len(commands)}]")
        run(cmd)
    return True


## 3. Validate The Environment

Run the project tests before stage-specific work. This confirms the installed code path is at least minimally healthy inside the Colab runtime.
                


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])
                


## 4. Optional Smoke Test

Use this cell only after you fill in `SMOKE_TEST_COMMANDS` in the config cell. Keep those commands tiny so you can catch path, dependency, and runtime issues before a full Stage 4: Unsupported-Confidence Control run.
                


In [ ]:
if RUN_SMOKE_TEST:
    ran_smoke = run_many(SMOKE_TEST_COMMANDS, label="smoke test")
    if not ran_smoke:
        print("Add one or more small commands to SMOKE_TEST_COMMANDS in the config cell.")
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.")
                


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Implementation Starts Here</strong><br/>
Sections 1-4 are setup and validation. Section 5 onward is the main Stage 4 work area.
<ul>
  <li><strong>Start here:</strong> create <code>src/keelnet/control.py</code> or an equivalent control module, then connect answer, support, and confidence signals in evaluation.</li>
  <li><strong>Finish here:</strong> confident unsupported answers go down without useful supported answers collapsing.</li>
  <li><strong>Out of scope:</strong> retrieval, adaptive balancing, unrelated model redesigns.</li>
</ul>
</div>


## 5. Run The Stage 4: Unsupported-Confidence Control Commands

This is the main Stage 4 implementation and run section. Everything before this point is setup, sync, or validation.

Fill in `STAGE_COMMANDS` in the config cell with the actual commands for this stage. Start with one command, make sure the outputs land in `OUTPUT_ROOT`, then add the rest.


In [ ]:
ran_stage = run_many(STAGE_COMMANDS, label="stage command")
if not ran_stage:
    print("Fill in STAGE_COMMANDS in the config cell before running this section.")
    print("Suggested module starting points:")
    for module_name in SUGGESTED_MODULES:
        print("-", module_name)
    print("Implementation hints:")
    for hint in IMPLEMENTATION_HINTS:
        print("-", hint)
                


## Stage Note Template

Keep your stage notes inside this notebook flow. Update them at three points:

1. before implementation: fill in the goal, success condition, and planned commands
2. after smoke test: record environment issues and command fixes
3. after a meaningful run: record metrics, verdict, and next actions

Use this structure for the generated run note:

- Run info
- Goal
- Commands
- Main metrics
- What changed
- What worked
- What failed or looks risky
- Error cases to review
- Decision
- Next actions


## 6. Save Notes And Review Artifacts

This cell creates teammate-friendly note files inside the Drive run folder and lists the current artifacts. Update the generated notes as you learn what does and does not work in Stage 4: Unsupported-Confidence Control.
                


In [ ]:
if not RUN_NOTES_FILE.exists():
    metric_lines = [f"- {metric}" for metric in TARGET_METRICS]
    RUN_NOTES_FILE.write_text(
        "\n".join(
            [
                f"# {STAGE_TITLE} Notes",
                "",
                "Update this note three times:",
                "1. before implementation: goal, success condition, and commands",
                "2. after smoke test: environment issues and command fixes",
                "3. after a meaningful run: metrics, verdict, and next actions",
                "",
                "## Run Info",
                f"- Branch: {CURRENT_BRANCH}",
                f"- `RUN_NAME`: {RUN_NAME}",
                f"- Output folder: {OUTPUT_ROOT}",
                "",
                "## Goal",
                "- One-sentence objective:",
                "- Success condition:",
                "- Out of scope:",
                "",
                "## Commands",
                "- Smoke test command(s):",
                "- Main command(s):",
                "- Input artifacts or checkpoints:",
                "- Output files to inspect:",
                "",
                "## Main Metrics",
                *metric_lines,
                "",
                "## What Changed",
                "- ",
                "",
                "## What Worked",
                "- ",
                "",
                "## What Failed Or Looks Risky",
                "- ",
                "",
                "## Error Cases To Review",
                "- ",
                "",
                "## Decision",
                "- Keep, revise, or stop:",
                "- Reason:",
                "",
                "## Next Actions",
                "1. ",
                "2. ",
                "3. ",
            ]
        )
        + "\n",
        encoding="utf-8",
    )

RUN_SUMMARY_FILE.write_text(
    json.dumps(
        {
            "stage": STAGE_TITLE,
            "run_name": RUN_NAME,
            "git_branch": CURRENT_BRANCH,
            "output_root": str(OUTPUT_ROOT),
            "target_metrics": TARGET_METRICS,
            "suggested_modules": SUGGESTED_MODULES,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


## What To Check Next

- Keep the notebook flow the same as Stage 1: edit in VS Code, push, rerun setup in Colab, then rerun only the cells you need.
- Save all run artifacts inside `OUTPUT_ROOT` so teammates can inspect them in Drive.
- Track these metrics before you call the stage successful: unsupported-answer rate, supported-answer rate, answer F1, abstain F1.
- Handoff condition: Do not move to the next stage until unsupported confident answers go down without answer quality collapsing.
                


## 7. Share This Run

This cell prints a minimal share-ready summary for teammates, saves it into the Drive run folder, and marks the run as completed so the next run becomes the next version.

Update the metric lines after you review the outputs for this stage.


In [ ]:
from datetime import datetime, timezone

share_lines = [
    f"# {STAGE_TITLE} Share Note",
    "",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    *[f"- {metric}: <fill in after review>" for metric in TARGET_METRICS],
    f"- Drive folder path: {OUTPUT_ROOT}",
]
share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
print(f"Update the metric lines in: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")
